# DLRS: free single-GPU CUDA validation

This notebook runs a small, deterministic check of an **installed DLRS wheel** on one CUDA GPU. It uses the runtime's existing PyTorch build, creates no distributed workers, and mounts no Drive.

Use the ordinary Colab notebook interface. Under **Runtime → Change runtime type**, select a **free GPU** if one is offered. Keep the standard memory setting. Do not purchase compute units, choose a paid/premium runtime, start an SSH connection, or switch to Enterprise. If free GPU access is unavailable, stop and retain the unavailable report. Google does not guarantee free GPU availability: [official Colab FAQ](https://research.google.com/colaboratory/faq.html).

The required gate covers 18 eager float32/FP16, optimizer-update, checkpoint, and gradient-accumulation tests. Two native BF16 tests are optional when the GPU lacks that capability. Passing these small checks is not evidence of workload nondegradation, scale, CUDA graphs, fused/capturable optimizers, or distributed/NCCL reliability.

Run the cells in order while present. The test deadline is five minutes, plus bounded setup/probing. Download the evidence before disconnecting. A terminated/deleted runtime can lose all local files; this notebook cannot recover them afterward.

In [ ]:
# This must be an immutable, reviewed commit containing the CUDA runner/tests.
REVISION = "2ff033d66d49b0adc994d30b00dfd23dfb6664d0"
REPOSITORY = "https://github.com/Thabhelo/pytorch-dlrs.git"

import datetime
import hashlib
import json
import os
import re
import subprocess
import sys
import tarfile
import uuid
from pathlib import Path

if not re.fullmatch(r"[0-9a-f]{40}", REVISION):
    raise ValueError("Set REVISION to the reviewed 40-character published commit before running.")
BASE = Path("/content") / ("dlrs-cuda-" + uuid.uuid4().hex[:10])
SOURCE = BASE / "source"
SETUP = BASE / "setup"
EVIDENCE = BASE / "evidence"
SETUP.mkdir(parents=True)
print("Evidence workspace:", BASE)
print("Pinned source:", REVISION)

Install only test/build tools, then build and install DLRS with dependency resolution disabled. No command installs or upgrades PyTorch. The recorded PyTorch version, CUDA build, and import path must remain identical before and after setup. Setup logs and the exact wheel are retained even if a later step fails.

In [ ]:
def setup_command(label, command, timeout=180):
    with (SETUP / (label + ".log")).open("w") as log:
        try:
            result = subprocess.run(command, stdout=log, stderr=subprocess.STDOUT,
                                    text=True, timeout=timeout, cwd=BASE)
        except subprocess.TimeoutExpired:
            log.write("\nSetup command timed out. Partial output retained.\n")
            raise
    if result.returncode:
        raise RuntimeError(f"Setup failed ({label}); download the evidence and inspect its log.")

TORCH_PROBE = "import json, torch; print(json.dumps({'version': str(torch.__version__), 'cuda_build': torch.version.cuda, 'path': torch.__file__}))"
setup_state = {"revision": REVISION, "status": "running", "started_utc": datetime.datetime.now(datetime.timezone.utc).isoformat()}
try:
    setup_command("torch-before", [sys.executable, "-I", "-c", TORCH_PROBE], timeout=60)
    torch_before = json.loads((SETUP / "torch-before.log").read_text())
    setup_command("git-init", ["git", "init", str(SOURCE)])
    setup_command("git-remote", ["git", "-C", str(SOURCE), "remote", "add", "origin", REPOSITORY])
    setup_command("git-fetch", ["git", "-C", str(SOURCE), "fetch", "--depth=1", "origin", REVISION])
    setup_command("git-checkout", ["git", "-C", str(SOURCE), "checkout", "--detach", "FETCH_HEAD"])
    setup_command("git-head", ["git", "-C", str(SOURCE), "rev-parse", "HEAD"])
    assert (SETUP / "git-head.log").read_text().strip() == REVISION
    setup_command("tools", [sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                            "pytest>=7,<9", "setuptools>=61", "wheel"])
    setup_command("build-wheel", [sys.executable, "-m", "pip", "wheel", "--no-deps", "--no-build-isolation",
                                  "--wheel-dir", str(SETUP / "wheels"), str(SOURCE)])
    wheels = list((SETUP / "wheels").glob("*.whl"))
    assert len(wheels) == 1, "Expected exactly one DLRS wheel"
    WHEEL = wheels[0]
    setup_command("install-wheel", [sys.executable, "-m", "pip", "install", "--no-deps", "--force-reinstall", str(WHEEL)])
    setup_command("torch-after", [sys.executable, "-I", "-c", TORCH_PROBE], timeout=60)
    torch_after = json.loads((SETUP / "torch-after.log").read_text())
    assert torch_before == torch_after, "PyTorch changed during setup; CUDA validation must stop"
    setup_state.update(status="ready", torch=torch_after, wheel=WHEEL.name,
                       wheel_sha256=hashlib.sha256(WHEEL.read_bytes()).hexdigest())
except BaseException as error:
    setup_state.update(status="failed_or_interrupted", error=repr(error))
    raise
finally:
    (SETUP / "setup.json").write_text(json.dumps(setup_state, indent=2) + "\n")
print("Ready. Existing PyTorch preserved:", torch_after)

Run only the single-process CUDA suite. The runner independently records environment and source identity, imports the installed package from site-packages, copies the reviewed tests outside the checkout, and rejects changed fingerprints or an incomplete test inventory. It never runs the repository's distributed suite.

If you interrupt this cell, run the download cell afterward. CUDA unavailable, a test failure, a timeout, or an interrupt never produces a passing gate.

In [ ]:
if setup_state["status"] != "ready":
    raise RuntimeError("Setup did not complete; use the download cell to retain its logs.")
command = [sys.executable, str(SOURCE / "benchmarks/validate_cuda.py"), "--installed-package",
           "--wheel", str(WHEEL), "--output", str(EVIDENCE), "--timeout", "300"]
print("Running 18 required CUDA checks and 2 optional native BF16 checks…", flush=True)
run_state = {"status": "running", "command": command}
with (SETUP / "runner-console.log").open("w") as log:
    process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, cwd=BASE)
    try:
        run_state["returncode"] = process.wait(timeout=420)
        run_state["status"] = "completed"
    except (KeyboardInterrupt, subprocess.TimeoutExpired) as error:
        run_state.update(status="interrupted_or_timed_out", error=type(error).__name__)
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait(timeout=10)
    finally:
        (SETUP / "notebook-run.json").write_text(json.dumps(run_state, indent=2) + "\n")
summary_path = EVIDENCE / "summary.json"
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print(json.dumps(summary, indent=2))
    passed = (run_state["status"] == "completed" and run_state.get("returncode") == 0
              and summary.get("status") == "passed" and summary.get("cuda_gate_passed") is True)
    print("Required CUDA gate:", "PASSED" if passed else "NOT PASSED")
else:
    print("CUDA gate NOT PASSED: no completed summary. Download the partial logs.")

Download the report even when setup/testing fails. The bundle includes setup logs, exact wheel, runner logs, GPU/driver/package/source identities, JUnit outcomes when available, and SHA-256 file manifests. It excludes the Git checkout, credentials, and environment-variable dumps. This download uses your browser; it does not upload to Drive or publish anything.

After the download completes, use **Runtime → Disconnect and delete runtime** to release the free GPU. Retain failures and skipped capabilities alongside successful runs.

In [ ]:
from google.colab import files

if "BASE" not in globals():
    raise RuntimeError("No evidence workspace was created; check the pinned revision cell.")
paths = sorted(path for directory in (SETUP, EVIDENCE) if directory.exists()
               for path in directory.rglob("*") if path.is_file())
manifest = {str(path.relative_to(BASE)): hashlib.sha256(path.read_bytes()).hexdigest() for path in paths}
manifest_path = BASE / "bundle-sha256.json"
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")
bundle = BASE / "dlrs-cuda-evidence.tar.gz"
with tarfile.open(bundle, "w:gz") as archive:
    for path in paths + [manifest_path]:
        archive.add(path, arcname=str(path.relative_to(BASE)), recursive=False)
print("Bundle:", bundle.name, "SHA-256:", hashlib.sha256(bundle.read_bytes()).hexdigest())
files.download(str(bundle))